In [2]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import illustris_python as il

# Ensure we import from /scripts (not notebooks/lg_analogues.py)
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root / "scripts"))
sys.path.insert(0, str(repo_root / "illustris_python"))

import lg_analogues as lg
from lg_abc import (
    build_pair_catalog,
    make_feature_matrix,
    make_theta_vector,
    abc_distance_diagonal,
    abc_rejection_mask,
    abc_kernel_weights,
    summarize_posterior,
    effective_sample_size,
    run_abc_catalog,
)


print("lg_analogues:", lg.__file__)


lg_analogues: /home/tp/Documents/uni/dsp/scripts/lg_analogues.py


In [4]:
# Data paths and fields
basePath = os.path.expanduser("~/Documents/uni/dsp/tng300/outputs")
snap = 99

subhalo_fields = [
    "SubhaloPos",
    "SubhaloVel",
    "SubhaloMassType",
    "SubhaloSFRinRad",
    "SubhaloFlag",
    "SubhaloGrNr",
]

sub = il.groupcat.loadSubhalos(basePath, snap, fields=subhalo_fields)
header_consts = lg.load_header_constants(basePath, snap)
h = header_consts["h"]
box_ckpch = header_consts["box_ckpch"]

print("Loaded subhalos:", sub["count"])
print("h:", h, "box_ckpch:", box_ckpch)

# Selection S: central + largest satellite in stellar-mass window
mstar_min = 2e10
mstar_max = 5e11

sample = lg.select_central_plus_largest_satellite_by_stellar_mass(
    sub=sub,
    basePath=basePath,
    snap=snap,
    h=h,
    mstar_min=mstar_min,
    mstar_max=mstar_max,
)

print("Selected objects:", sample.keep_idx.size)

# Pair finding in separation window
r_min_kpc = 500.0
r_max_kpc = 1000.0

pair_set = lg.find_pairs_periodic(
    pos=sample.pos,
    vel=sample.vel,
    grnr=sample.grnr,
    h=h,
    box_ckpch=box_ckpch,
    r_min_kpc=r_min_kpc,
    r_max_kpc=r_max_kpc,
)

print("Pairs before filtering:", pair_set.i.size)
print("same-host pairs:", int(np.sum(pair_set.same_host)))
print("different-host pairs:", int(np.sum(~pair_set.same_host)))

# Pipeline filters: isolation (no third object with M* >= x * max(M*_pair) in relevant FoF group(s))
pipeline = lg.AnaloguePipeline(pair_set)

isolation_x = 1.5
iso_mask = lg.isolation_filter_no_third_factor_x_in_same_group(
    sub=sub,
    pair_i=pipeline.pairs.i,
    pair_j=pipeline.pairs.j,
    sample_grnr=sample.grnr,
    sample_keep_idx_global=sample.keep_idx,
    h=h,
    x=isolation_x,
)
pipeline.apply_filter(f"isolation_x{isolation_x}", iso_mask)

pipeline.print_cutflow()
cutflow = pipeline.get_cutflow()
cutflow


Loaded subhalos: 14485709
h: 0.6774 box_ckpch: 205000.0
Selected objects: 45392
Pairs before filtering: 2655
same-host pairs: 2084
different-host pairs: 571
initial: 2655 pairs remaining
isolation_x1.5: 2601 pairs remaining


[{'label': 'initial', 'before': 2655, 'after': 2655, 'frac_kept': 1.0},
 {'label': 'isolation_x1.5',
  'before': 2655,
  'after': 2601,
  'frac_kept': 0.9796610169491525}]

In [6]:
from lg_abc import build_pair_catalog
from lg_gmm_kde import prepare_density_problem, run_kde_conditioning, fit_joint_gmm, condition_gmm

cat = build_pair_catalog(sample=sample, pairs=pipeline.pairs, sub=sub, sfr_field="SubhaloSFRinRad")

features = ["r_kpc", "v_r", "v_t"]
target = "mdm_sum"
x_obs = np.array([770.0, -109.0, 17.0], dtype=float)
sigma = np.array([50.0, 20.0, 30.0], dtype=float)

problem = prepare_density_problem(cat, features=features, target=target, target_log10=True)
kde_result = run_kde_conditioning(problem["X"], problem["theta"], x_obs=x_obs, bandwidth=sigma)

gmm_fit = fit_joint_gmm(problem["X"], problem["theta"], n_components="bic", random_state=0)
gmm_result = condition_gmm(gmm_fit, x_obs=x_obs, n_samples=20000, random_state=0)

print(kde_result)

{'method': 'kde', 'weights': array([1.80184806e-19, 3.88223964e-70, 3.19728683e-15, ...,
       1.59729975e-51, 8.75041807e-07, 2.81003005e-06], shape=(2601,)), 'distance': array([ 8.94823411, 17.7030894 ,  7.77838245, ..., 15.08898062,
        4.65277059,  4.3948739 ], shape=(2601,)), 'summary': {'mean': 12.520913276048555, 'median': 12.472528060017112, 'q16': 12.369241135213237, 'q84': 12.609286523989482, 'ess': 55.290475339145125}, 'theta_grid': array([11.5827102 , 11.58967088, 11.59663155, 11.60359222, 11.6105529 ,
       11.61751357, 11.62447425, 11.63143492, 11.6383956 , 11.64535627,
       11.65231694, 11.65927762, 11.66623829, 11.67319897, 11.68015964,
       11.68712032, 11.69408099, 11.70104166, 11.70800234, 11.71496301,
       11.72192369, 11.72888436, 11.73584504, 11.74280571, 11.74976638,
       11.75672706, 11.76368773, 11.77064841, 11.77760908, 11.78456976,
       11.79153043, 11.7984911 , 11.80545178, 11.81241245, 11.81937313,
       11.8263338 , 11.83329448, 11.8402551